# 04 · Géolocalisation des hôtels Booking

Objectif : donner une latitude et une longitude à chacun des 1 705 hôtels Booking, pour les placer sur la carte comme les annonces Airbnb et les rattacher à un quartier par jointure spatiale.

Sortie : `data/external/hotels_geocodage.csv`, une ligne par hôtel (contrat dans `docs/handoff_geocodage_hotels.md`, section 4). Seules les lignes `fiable = true` sont reprises dans la table commune.

Chaîne, de la méthode la plus sûre à la plus coûteuse (logique dans `src/prep/geocodage.py`) :

1. **OSM (Overpass)** : une requête par ville, appariement flou des noms ;
2. **Nominatim** : géocodage `"<nom>, <ville>"` pour les hôtels restants (1 requête/s, cache local) ;
3. **corrections relues** (`data/external/hotels_geocodage_corrections.csv`) pour les cas que les deux premières méthodes ne règlent pas.

Les réponses des services sont en cache sous `data/external/cache/` (non versionné) : relancer le notebook ne refait aucune requête déjà faite.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.prep import geocodage as g, hotels

pd.set_option("display.max_colwidth", 60)
brut = hotels.dedoublonner(hotels.charger_brut("../data/raw/hotels"))
len(brut)

## 1. Normalisation et similarité des noms

Les noms sont mis en minuscules, sans accents ni ponctuation, et privés des mots génériques (« hotel », « the », le nom de la ville…).

La similarité combine :
- un **Jaccard pondéré par l'IDF** des mots (80 %) : un mot fréquent dans la ville (« kensington », « garden ») pèse peu, un mot rare (« amano ») pèse beaucoup. Sans cette pondération, « AMANO Covent Garden » s'appariait à « Covent Garden Hotel » ;
- `token_sort_ratio` de rapidfuzz (20 %), pour les petites différences de graphie.

In [ ]:
for a, b in [("AMANO Covent Garden", "Covent Garden Hotel"),
             ("Holiday Inn London - Whitechapel, an IHG Hotel", "Holiday Inn London - Whitechapel"),
             ("ibis London Stratford", "ibis Styles London Excel")]:
    print(f"{g.similarite(g.normaliser_nom(a), g.normaliser_nom(b)):5.1f}  {a}  /  {b}")

## 2. Exécution de la chaîne

`geocoder` enchaîne OSM, Nominatim et les corrections relues, et calcule les contrôles de la section 5 du handoff pour chaque candidat.

In [ ]:
out, centres, candidats, info = g.geocoder(brut)
out = out.merge(info[["link", "title", "location"]], on="link")

## 3. Le « centre » de Booking

Booking ne documente pas le point depuis lequel il calcule `distanceFromCenter`. On l'estime à partir des appariements OSM les plus sûrs (similarité ≥ 97, quartier cohérent) : c'est le point qui minimise l'écart médian entre la distance recalculée (haversine) et la distance annoncée.

L'écart médian tombe à 20-30 m, à peu près l'arrondi de Booking : la distance annoncée est une distance à vol d'oiseau depuis un point fixe. On le retrouve à quelques mètres près.

In [ ]:
pd.DataFrame([{"ville": v, "lat": la, "lon": lo} for v, (la, lo) in centres.items()]).round(5)

Ces points correspondent à des repères connus : Hôtel de Ville (Paris), Charing Cross et Trafalgar Square (Londres), porte de Brandebourg et Reichstag (Berlin), piazza Venezia (Rome). C'est utile pour le rapport, qui laisse ouverte la question de la définition du centre : il faudra vérifier si le centre Airbnb est le même.

## 4. Paliers d'acceptation

| Méthode | Similarité | Exigence supplémentaire |
|---|---|---|
| OSM | ≥ 90 | aucune pour l'acceptation ; contrôles pour `fiable` |
| OSM | ≥ 75 | écart de distance ≤ 0,5 km et quartier cohérent |
| OSM | ≥ 60 | écart ≤ 0,15 km et quartier cohérent |
| Nominatim | ≥ 60 | écart ≤ 0,5 km et quartier cohérent, catégorie `tourism`, `amenity`, `building` ou `leisure` |
| Nominatim | ≥ 45 | écart ≤ 0,15 km, mêmes conditions |

Un objet OSM revendiqué par plusieurs hôtels reste à celui qui a la meilleure similarité ; les autres passent à leur candidat suivant.

In [ ]:
ok = candidats[(candidats["methode"] == "osm") & (candidats["similarite"] >= 97)]
ok = ok.sort_values("similarite", ascending=False).drop_duplicates("link")
print(ok["ecart_dist_km"].quantile([.5, .9, .95, .99]).round(2))
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(ok["ecart_dist_km"].clip(upper=2), bins=80)
ax.axvline(g.TOLERANCE_KM, color="k", lw=1, ls="--")
ax.set(xlabel="écart |distance recalculée − distance Booking| (km, écrêté à 2)", ylabel="hôtels",
       title="Appariements OSM sûrs : écart de distance au centre")
plt.show()

## 5. Contrôles et résultats

`fiable` réunit quatre conditions :
- point dans l'emprise de la ville ;
- écart de distance ≤ 0,5 km ;
- point dans le contour du quartier Booking, à 200 m près (sauf Rome, où le contrôle est seulement indicatif) ;
- coordonnées non partagées avec un hôtel mieux apparié.

In [ ]:
taux = out.groupby("ville_brut").agg(hotels=("link", "size"),
                                     geolocalises=("lat", lambda s: s.notna().mean()),
                                     fiables=("fiable", "mean"))
taux.loc["total"] = [len(out), out["lat"].notna().mean(), out["fiable"].mean()]
taux.style.format({"geolocalises": "{:.1%}", "fiables": "{:.1%}"})

In [ ]:
pd.crosstab(out["ville_brut"], out["methode"].fillna("non trouvé"), margins=True)

In [ ]:
pd.crosstab(out.loc[out["fiable"], "ville_brut"], out.loc[out["fiable"], "methode"], margins=True)

### Contrôle de quartier

Part des points fiables qui tombent dans le contour du quartier Booking. À Rome, ce contrôle reste indicatif : les noms Booking (« Spagna », « Central Station ») ne correspondent qu'approximativement aux rioni et quartieri.

In [ ]:
out.groupby("ville_brut")["controle_quartier"].value_counts(dropna=False).unstack()

### Doublons de coordonnées

In [ ]:
f = out[out["fiable"]]
dup = f[f.duplicated(["lat", "lon"], keep=False)].sort_values(["lat", "lon"])
dup[["ville_brut", "title", "nom_trouve", "lat", "lon"]]

### Hôtels sans point fiable

In [ ]:
out.loc[~out["fiable"], ["ville_brut", "title", "location", "methode", "nom_trouve", "similarite",
                         "ecart_dist_km", "controle_quartier"]]

## 6. Contrôle visuel

Les points fiables sur les contours de quartiers, avec le centre Booking estimé.

In [ ]:
from shapely.geometry import MultiPolygon

fig, axes = plt.subplots(2, 2, figsize=(12, 11))
for ax, v in zip(axes.flat, ["london", "berlin", "paris", "rome"]):
    for geom in g.charger_contours(v).values():
        for p in (geom.geoms if isinstance(geom, MultiPolygon) else [geom]):
            ax.plot(*p.exterior.xy, color="0.7", lw=0.5)
    d = out[(out["ville_brut"] == v) & out["fiable"]]
    for m, col in [("osm", "tab:blue"), ("nominatim", "tab:orange"), ("manuel", "tab:red")]:
        s = d[d["methode"] == m]
        ax.scatter(s["lon"], s["lat"], s=6, color=col, label=f"{m} ({len(s)})")
    ax.scatter(centres[v][1], centres[v][0], marker="*", s=150, color="k", label="centre estimé")
    d = out[out["ville_brut"] == v]
    ax.set(title=v, xlim=(d["lon"].quantile(.005) - .02, d["lon"].quantile(.995) + .02),
           ylim=(d["lat"].quantile(.005) - .02, d["lat"].quantile(.995) + .02))
    ax.set_aspect(1 / np.cos(np.radians(centres[v][0])))
    ax.legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.show()

## 7. Export

In [ ]:
g.ecrire(out, Path("../data/external/hotels_geocodage.csv"))
verif = pd.read_csv("../data/external/hotels_geocodage.csv")
assert len(verif) == len(brut) == verif["link"].nunique()
assert list(verif.columns) == g.COLONNES_SORTIE
verif.head()